# 02. Preprocessing & Windowing - NinaPro DB1 (Subject 1)
**Objective:** Filter raw data, normalize amplitudes, extract overlapping windows, solve class imbalance, and serialize to HDF5.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np

sys.path.append(os.path.abspath('../'))
from src.utils import load_ninapro_mat
from src.preprocess import sEMGPreprocessor, save_preprocessed_hdf5

### 1. Load Data with Label Offsets
Replicating our robust loading logic from the EDA phase.

In [2]:
db_dir = "../data/raw/Ninapro_DB1/s1"
exercises = ['E1', 'E2', 'E3']
offsets = {'E1': 0, 'E2': 12, 'E3': 29} 

emg_list, labels_list, reps_list = [], [], []

for ex in exercises:
    mat_path = os.path.join(db_dir, f"S1_A1_{ex}.mat")
    data = load_ninapro_mat(mat_path)
    
    labels = data['labels'].copy()
    active_mask = labels > 0
    labels[active_mask] += offsets[ex]
    
    emg_list.append(data['emg'])
    labels_list.append(labels)
    reps_list.append(data['reps'])

emg_full = np.vstack(emg_list)
labels_full = np.concatenate(labels_list)
reps_full = np.concatenate(reps_list)

print(f"Raw Global Shape: {emg_full.shape}")

Raw Global Shape: (471483, 10)


### 2. Filter & Standardize
Applying safety-checked bandpass filters and Z-score normalization.

In [3]:
preprocessor = sEMGPreprocessor(database_name="DB1")

# Apply filters
filtered_emg = preprocessor.filter_signal(emg_full)
print("Filtering complete.")

# Z-score standardization
norm_emg = preprocessor.standardize(filtered_emg)
print("Standardization complete.")

Filtering complete.
Standardization complete.


### 3. Sliding Windows & Class Balancing
Slicing the continuous signal into 200ms windows with 50% overlap, followed by undersampling the Rest class.

In [4]:
print("Extracting sliding windows...")
X_win, y_win, reps_win = preprocessor.extract_windows(norm_emg, labels_full, reps_full)
print(f"Total Windows Extracted: {X_win.shape}")

print("\nBalancing Rest Class...")
X_bal, y_bal, reps_bal = preprocessor.balance_rest_class(X_win, y_win, reps_win)

# Verification
unique, counts = np.unique(y_bal, return_counts=True)
print(f"Balanced Dataset Shape: {X_bal.shape}")
print(f"Rest Class Windows (0): {counts[0]}")
print(f"Average Active Gesture Windows: {int(np.mean(counts[1:]))}")

Extracting sliding windows...
Total Windows Extracted: (47147, 20, 10)

Balancing Rest Class...
Balanced Dataset Shape: (18630, 20, 10)
Rest Class Windows (0): 522
Average Active Gesture Windows: 348


### 4. Serialize to HDF5
Saving the processed tensors to disk for fast loading during model training.

In [5]:
print("Serializing to HDF5...")
save_preprocessed_hdf5(subject_id=1, X=X_bal, y=y_bal, reps=reps_bal, db_name="DB1")
print("Pipeline complete! File saved in /data/preprocessed/")

Serializing to HDF5...
Pipeline complete! File saved in /data/preprocessed/
